In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

# 1. 브라우저 설정
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

TARGET_URL = "https://www.mk.co.kr/search/news?word=%EC%A3%BC%EC%A3%BC%ED%99%98%EC%9B%90&sort=desc&dateType=direct&startDate=2019-01-01&endDate=2025-12-31&searchField=all&newsType=all"

def mk_ultimate_crawl(url):
    driver.get(url)
    wait = WebDriverWait(driver, 20)
    
    print("🌐 페이지 접속 완료. 기사가 나타날 때까지 대기합니다...")
    
    # [중요] 기사 제목이 하나라도 나타날 때까지 최대 20초 대기
    try:
        wait.until(EC.presence_of_element_located((By.XPATH, "//h3")))
        print("✅ 기사 요소를 확인했습니다. 수집을 시작합니다.")
    except:
        print("❌ 기사 요소를 찾지 못했습니다. URL을 다시 확인하거나 페이지가 로딩되지 않았습니다.")
        return []

    final_results = []
    seen_titles = set()

    try:
        while True:
            # [Step 1] 기사 덩어리들을 광범위하게 수집 (li 태그 전체)
            # 클래스명이 바뀌어도 대응할 수 있게 XPATH 사용
            articles = driver.find_elements(By.XPATH, "//li[descendant::h3]")
            
            for art in articles:
                try:
                    # h3 태그는 제목
                    title = art.find_element(By.TAG_NAME, "h3").text.strip()
                    
                    # 날짜는 오른쪽의 큰 날짜 또는 span 태그 탐색
                    # 이미지상 우측 12.31 2025는 보통 'area_date'나 'time_txt' 클래스임
                    # 클래스 무관하게 텍스트 내에 숫자가 포함된 요소를 찾음
                    date_elements = art.find_elements(By.XPATH, ".//*[contains(@class, 'date') or contains(@class, 'time')]")
                    date = date_elements[0].text.replace('\n', ' ').strip() if date_elements else "날짜없음"
                    
                    if title and title not in seen_titles:
                        final_results.append([title, date])
                        seen_titles.add(title)
                except:
                    continue
            
            print(f"📊 현재 수집 완료: {len(final_results)}건", end='\r')

            # [Step 2] 천천히 스크롤 내리기
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.5)

            # [Step 3] '더보기' 버튼 클릭 (텍스트로 찾기)
            try:
                # '더보기'라는 글자가 포함된 버튼을 찾아 클릭
                more_btn = driver.find_element(By.XPATH, "//button[contains(., '더보기')]")
                
                if more_btn.is_displayed():
                    driver.execute_script("arguments[0].click();", more_btn)
                    time.sleep(2.0) # 로딩 대기
                else:
                    # 버튼은 있는데 화면에 안 보이면 스크롤 더 내리기
                    driver.execute_script("window.scrollBy(0, 500);")
                    time.sleep(1.0)
            except:
                # 버튼을 아예 못 찾으면 스크롤 한 번 더 해보고 종료 판단
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(1.5)
                try:
                    more_btn = driver.find_element(By.XPATH, "//button[contains(., '더보기')]")
                    driver.execute_script("arguments[0].click();", more_btn)
                except:
                    print(f"\n🔚 더 이상 '더보기' 버튼이 없습니다. 최종 {len(final_results)}건 수집.")
                    break
            
            # 500개마다 백업
            if len(final_results) > 0 and len(final_results) % 500 == 0:
                pd.DataFrame(final_results, columns=['제목', '날짜']).to_csv("mk_dividend_backup.csv", index=False, encoding="utf-8-sig")

    except Exception as e:
        print(f"\n❌ 실행 중 오류 발생: {e}")

    return final_results

try:
    results = mk_ultimate_crawl(TARGET_URL)
    if results:
        df = pd.DataFrame(results, columns=['제목', '날짜'])
        df.to_csv("매일경제_주주환원_4523건_도전.csv", index=False, encoding="utf-8-sig")
        print(f"🎉 성공! 파일 저장됨: 매일경제_배당_17000건_도전.csv")
    else:
        print("😭 여전히 0건입니다. 브라우저가 기사를 다 불러오기 전에 코드가 끝났을 수 있습니다.")
finally:
    driver.quit()

🌐 페이지 접속 완료. 기사가 나타날 때까지 대기합니다...
✅ 기사 요소를 확인했습니다. 수집을 시작합니다.
📊 현재 수집 완료: 99건

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

# 1. 브라우저 설정
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# 2. 수집할 기간 리스트 (6개월 단위 쪼개기)
# 2025년부터 2019년까지 역순으로 구성
date_ranges = [
    ("2025-06-30", "2025-12-31"), ("2024-12-31", "2025-06-30"),
    ("2024-06-30", "2024-12-31"), ("2023-12-31", "2024-06-30"),
    ("2023-06-30", "2023-12-31"), ("2022-12-31", "2023-06-30"),
    ("2022-06-30", "2022-12-31"), ("2021-12-31", "2022-06-30"),
    ("2021-06-30", "2021-12-31"), ("2020-12-31", "2021-06-30"),
    ("2020-06-30", "2020-12-31"), ("2019-12-31", "2020-06-30"),
    ("2019-06-30", "2019-12-31"), ("2018-12-31", "2019-06-30")
]

def collect_period_data(start_date, end_date):
    keyword = "주주환원" # 또는 'pbr'
    url = f"https://www.mk.co.kr/search/news?word={keyword}&sort=desc&dateType=direct&startDate={start_date}&endDate={end_date}&searchField=all&newsType=all"
    
    driver.get(url)
    time.sleep(6) # 첫 로딩은 더 넉넉하게
    
    period_results = []
    seen_titles = set()
    prev_count = 0
    no_growth_count = 0
    
    print(f"📅 수집 기간: {start_date} ~ {end_date}")

    while True:
        # [Step 1] 스크롤을 아주 천천히 여러 번 나눠서 내리기 (매경의 동적 로딩 대응)
        for _ in range(3): 
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(0.5)
        
        # [Step 2] 데이터 추출 (더 꼼꼼하게)
        articles = driver.find_elements(By.XPATH, "//li[descendant::h3]")
        for art in articles:
            try:
                # 제목이 화면에 로드될 때까지 기다리는 대신, 텍스트가 비어있으면 건너뜀
                title_el = art.find_element(By.TAG_NAME, "h3")
                title = title_el.text.strip()
                
                if not title: continue # 로딩 안 된 제목 패스

                date_els = art.find_elements(By.XPATH, ".//*[contains(@class, 'date') or contains(@class, 'time')]")
                date = date_els[0].text.replace('\n', ' ').strip() if date_els else "날짜없음"
                
                if title not in seen_titles:
                    period_results.append([title, date])
                    seen_titles.add(title)
            except: continue
        
        current_count = len(period_results)
        print(f"   📊 진행 상황: {current_count}건 수집 중...", end='\r')

        # [Step 3] 종료 판단 (더 보수적으로 체크)
        if current_count == prev_count:
            no_growth_count += 1
        else:
            no_growth_count = 0
            prev_count = current_count

        # 데이터가 안 늘어날 때, 스크롤을 끝까지 한 번 더 내려보고 그래도 없으면 종료
        if no_growth_count >= 4: 
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(3) # 마지막 확인 대기
            if len(driver.find_elements(By.XPATH, "//li[descendant::h3]")) <= len(articles):
                print(f"\n   ✅ 기간 완료: {current_count}건 수집됨.")
                break
            else:
                no_growth_count = 0 # 데이터가 더 있으면 다시 재개

        # [Step 4] 더보기 클릭
        try:
            more_btn = driver.find_element(By.XPATH, "//button[contains(., '더보기')]")
            if more_btn.is_displayed():
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", more_btn)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(3) # 기사가 1,000건 넘어가면 로딩이 매우 느려지므로 3초 대기
        except:
            time.sleep(2)
            continue
            
    return period_results
# 3. 메인 실행부
all_collected_data = []

try:
    for start, end in date_ranges:
        period_data = collect_period_data(start, end)
        all_collected_data.extend(period_data)
        
        # 각 기간 끝날 때마다 '누적 파일' 업데이트 (안전장치)
        temp_df = pd.DataFrame(all_collected_data, columns=['제목', '날짜'])
        temp_df.to_csv("매일경제_주주환원_통합데이터.csv", index=False, encoding="utf-8-sig")
        print(f"💾 현재까지 총 {len(temp_df)}건 누적 저장 완료.")
        print("-" * 40)
        time.sleep(3) # 서버 과부하 방지를 위한 휴식

    print(f"🎉 전 기간 수집 완료! 총 {len(all_collected_data)}건의 데이터를 확보했습니다.")

except Exception as e:
    print(f"\n❌ 실행 도중 오류 발생: {e}")

finally:
    driver.quit()

📅 수집 기간: 2025-06-30 ~ 2025-12-31
   📊 진행 상황: 579건 수집 중...
   ✅ 기간 완료: 579건 수집됨.
💾 현재까지 총 579건 누적 저장 완료.
----------------------------------------
📅 수집 기간: 2024-12-31 ~ 2025-06-30
   📊 진행 상황: 593건 수집 중...
   ✅ 기간 완료: 593건 수집됨.
💾 현재까지 총 1172건 누적 저장 완료.
----------------------------------------
📅 수집 기간: 2024-06-30 ~ 2024-12-31
   📊 진행 상황: 719건 수집 중...
   ✅ 기간 완료: 719건 수집됨.
💾 현재까지 총 1891건 누적 저장 완료.
----------------------------------------
📅 수집 기간: 2023-12-31 ~ 2024-06-30
   📊 진행 상황: 914건 수집 중...
   ✅ 기간 완료: 914건 수집됨.
💾 현재까지 총 2805건 누적 저장 완료.
----------------------------------------
📅 수집 기간: 2023-06-30 ~ 2023-12-31
   📊 진행 상황: 290건 수집 중...
   ✅ 기간 완료: 290건 수집됨.
💾 현재까지 총 3095건 누적 저장 완료.
----------------------------------------
📅 수집 기간: 2022-12-31 ~ 2023-06-30
   📊 진행 상황: 361건 수집 중...
   ✅ 기간 완료: 361건 수집됨.
💾 현재까지 총 3456건 누적 저장 완료.
----------------------------------------
📅 수집 기간: 2022-06-30 ~ 2022-12-31
   📊 진행 상황: 150건 수집 중...
   ✅ 기간 완료: 150건 수집됨.
💾 현재까지 총 3606건 누적 저장 완료.
--------------------